# 03b. 터널 입구 시험 사진 더 받기 (Mapillary)

**왜**: 지금 주 판정은 Mapillary 입구 66장(51장면)뿐이라 점수가 많이 흔들려요. 학습에 안 쓴 입구 사진을 더 모아 시험지를 키워요.

**순서** (이 노트북은 **세기 → 확인 → 받기** 순서. 처음에는 1~3만 돌려요)
1. 이미 받은 목록(`mapillary_raw.jsonl`, 고속도로 입구 2,654곳 주변 ±170 m)으로 **조건을 조금 넓혀** 몇 장 더 나오는지 세기 (인터넷 안 씀, 몇 초)
2. (선택) 고속도로가 아닌 **일반 도로터널**(국도·지방도 등, 지하차도 제외) 입구 주변도 목록만 받아 세기 (인터넷 씀, 15~30분)
3. 보고 표 출력
4. (확인한 뒤) 사진 받기: 입구 × 촬영 날마다 1장, 이미 받은 172장은 빼고. 받은 사진은 `사진분류_도구.html`로 분류

**준비**: `sunsight_data/.env`의 `MAPILLARY_TOKEN` (03과 같음). 실행 위치 `머신러닝3조/sunsight_video`


In [ ]:
import os, math, time, json
from pathlib import Path
import numpy as np, pandas as pd, requests

ROOT = Path('..') / 'sunsight_data'
OUT = Path('data')
def read_env(p):
    env = {}
    if p.exists():
        for line in p.read_text(encoding='utf-8').splitlines():
            if '=' in line and not line.strip().startswith('#'):
                k, v = line.split('=', 1); env[k.strip()] = v.strip().strip('"')
    return env
TOKEN = read_env(ROOT / '.env').get('MAPILLARY_TOKEN') or os.environ.get('MAPILLARY_TOKEN')
assert TOKEN, 'sunsight_data/.env 에 MAPILLARY_TOKEN=... 이 필요해요'

def bearing(lat1, lon1, lat2, lon2):
    p1, p2 = math.radians(lat1), math.radians(lat2); dl = math.radians(lon2 - lon1)
    x = math.sin(dl) * math.cos(p2); y = math.cos(p1) * math.sin(p2) - math.sin(p1) * math.cos(p2) * math.cos(dl)
    return (math.degrees(math.atan2(x, y)) + 360) % 360
def dist_m(lat1, lon1, lat2, lon2):
    return 6371000 * math.hypot(math.radians(lat2 - lat1), math.radians(lon2 - lon1) * math.cos(math.radians(lat1)))

# 이미 받은 사진 (03에서 받은 172장) → 다시 받지 않음
have = set(p.stem for p in (OUT / 'tunnel_photos').glob('*.jpg'))
print('이미 받은 사진', len(have), '장')


## 1. 고속도로 입구: 조건 넓혀 세기 (인터넷 안 씀)

| 조건 | 03 (지금) | 03b (넓힘) | 이유 |
|---|---|---|---|
| 입구까지 거리 | 20~150 m | **10~170 m** | 받아 둔 목록이 ±170 m라 그 안에서 최대 |
| 카메라 방향 차이 | 30° 이하 | **45° 이하** | 분류 기준 "입구가 옆으로 비껴 있어도 앞에 보이면 입구"와 맞춤 |
| 시각 | 7~18시 | 7~18시 (같음) | 해 낮을 때는 정답 일사량이 거의 0 |

같은 입구를 같은 날 찍은 사진은 거의 같은 사진이라 **입구 × 촬영 날** 하나를 1장으로 세요.


In [ ]:
def make_portals(tun, la1, lo1, la2, lo2, name):
    P = []
    for i, r in tun.iterrows():
        for side, la, lo in [('시작', la1, lo1), ('종료', la2, lo2)]:
            if pd.notna(r[la]) and pd.notna(r[lo]):
                P.append({'tunnel': r[name], 'side': side, 'lat': float(r[la]), 'lon': float(r[lo])})
    return pd.DataFrame(P)

def filter_raw(raw_path, portals, dmin=10, dmax=170, ang=45, h1=7, h2=18):
    rows = []
    for line in open(raw_path, encoding='utf-8'):
        rec = json.loads(line); p = portals.loc[rec['portal']]
        for im in rec['images']:
            g = (im.get('computed_geometry') or im.get('geometry') or {}).get('coordinates')
            head = im.get('computed_compass_angle', im.get('compass_angle'))
            if not g or head is None or im.get('is_pano'): continue
            lon, lat = g
            d = dist_m(lat, lon, p.lat, p.lon)
            diff = abs((bearing(lat, lon, p.lat, p.lon) - head + 180) % 360 - 180)
            rows.append({'portal': rec['portal'], 'tunnel': p.tunnel, 'side': p.side, 'image_id': str(im['id']),
                         'captured_at': pd.to_datetime(im['captured_at'], unit='ms', utc=True).tz_convert('Asia/Seoul'),
                         'lat': lat, 'lon': lon, 'heading': head, 'dist_m': round(d, 1), 'angle_diff': round(diff, 1),
                         'sequence': im.get('sequence')})
    img = pd.DataFrame(rows)
    img['hour'] = img.captured_at.dt.hour
    c = img[img.dist_m.between(dmin, dmax) & (img.angle_diff <= ang) & img.hour.between(h1, h2)].copy()
    c['date'] = c.captured_at.dt.date; c['month'] = c.captured_at.dt.month
    return c

tun_hw = pd.read_csv(ROOT / 'data' / '2_중간' / 'tunnel_expressway.csv', encoding='utf-8-sig')
port_hw = make_portals(tun_hw, 'start_lat', 'start_lon', 'end_lat', 'end_lon', '터널명')
cand_hw = filter_raw(OUT / 'mapillary_raw.jsonl', port_hw)
cand_hw['도로'] = '고속도로'
pick_hw = cand_hw.sort_values('dist_m').drop_duplicates(['portal', 'date'])
pick_hw['이미받음'] = pick_hw.image_id.isin(have)
# 이미 받은 172장과 같은 입구 × 같은 날이면 새 장면이 아님 → 뺌
old_keys = set(map(tuple, cand_hw[cand_hw.image_id.isin(have)][['portal', 'date']].values))
pick_hw['새 장면'] = [k not in old_keys for k in map(tuple, pick_hw[['portal', 'date']].values)]
print('고속도로: 후보 사진', len(cand_hw), '| 입구 × 날', len(pick_hw), '| 그중 새 장면(아직 안 받음)', int(pick_hw['새 장면'].sum()))


## 2. (선택) 일반 도로터널도 세기 (인터넷 씀, 15~30분)

고속도로가 아닌 도로터널(국도·지방도·시도 등) 입구 주변 사진 목록만 받아요. **사진 파일은 안 받아요.** 지하차도는 뺍니다.

⚠️ 주의: 우리 터널 안 밝기 280은 **100 km/h 고속도로** 지침값이에요. 국도(80 km/h 이하)는 지침 계수가 0.05 이하라 터널 안 밝기가 달라요. 그래서 일반 도로터널 사진은 **"같은 공식이라면"이라는 참고 시험**으로만 쓸 수 있어요. 쓸지 정한 뒤 받기로 해요.

`COUNT_OTHER = True`로 바꾸면 실행돼요. 중간에 끊겨도 이어서 받아요.


In [ ]:
COUNT_OTHER = False   # True로 바꾸면 일반 도로터널 목록을 받아 셈

tun_all = pd.read_csv(ROOT / 'data' / '1_원본' / 'tunnel.csv', encoding='utf-8-sig')
tun_ot = tun_all[(tun_all['도로종류'] != '고속국도') & (tun_all['터널종류'] == '도로터널')].reset_index(drop=True)
port_ot = make_portals(tun_ot, '터널시작점위도', '터널시작점경도', '터널종료점위도', '터널종료점경도', '터널명')
print('일반 도로터널', len(tun_ot), '개 · 입구', len(port_ot), '곳', tun_ot['도로종류'].value_counts().to_dict())

RAW2 = OUT / 'mapillary_raw_일반도로.jsonl'
FIELDS = 'id,captured_at,compass_angle,computed_compass_angle,geometry,computed_geometry,is_pano,sequence'
D = 0.0015   # ≈ 167 m (03과 같음)
if COUNT_OTHER:
    done = set()
    if RAW2.exists():
        for line in RAW2.open(encoding='utf-8'): done.add(json.loads(line)['portal'])
    with RAW2.open('a', encoding='utf-8') as f:
        for k, p in port_ot.iterrows():
            if k in done: continue
            dlon = D / math.cos(math.radians(p.lat))
            bbox = f'{p.lon-dlon},{p.lat-D},{p.lon+dlon},{p.lat+D}'
            data = None
            for attempt in range(3):
                try:
                    r = requests.get('https://graph.mapillary.com/images',
                                     params={'access_token': TOKEN, 'bbox': bbox, 'fields': FIELDS, 'limit': 2000}, timeout=30)
                    r.raise_for_status(); data = r.json().get('data', []); break
                except Exception as e:
                    print(k, '재시도', e); time.sleep(2 ** attempt)
            if data is None: continue
            f.write(json.dumps({'portal': int(k), 'images': data}, ensure_ascii=False) + '\n')
            if k % 200 == 0: print(k, '/', len(port_ot))
            time.sleep(0.05)
    print('완료')

pick_ot = pd.DataFrame()
if RAW2.exists():
    cand_ot = filter_raw(RAW2, port_ot); cand_ot['도로'] = '일반 도로터널'
    pick_ot = cand_ot.sort_values('dist_m').drop_duplicates(['portal', 'date']).copy()
    pick_ot['이미받음'] = False; pick_ot['새 장면'] = True
    print('일반 도로터널: 후보 사진', len(cand_ot), '| 입구 × 날', len(pick_ot))


## 3. 보고 표 (이 칸 출력을 Claude에게 붙여 주세요)

- **예상 입구 사진 수** = 새 장면 × 0.41. 03에서 받은 172장 중 "입구"로 분류한 게 70장(41%)이었어서 같은 비율로 어림잡아요.
- 달이 고르게 퍼질수록 좋아요 (지금 시험은 12월 22장 · 5월 28장에 몰려 있음).


In [ ]:
picks = pd.concat([pick_hw, pick_ot], ignore_index=True)
new = picks[picks['새 장면']]
tab = new.groupby('도로').agg(입구x날=('image_id', 'size'), 입구=('portal', 'nunique'), 터널=('tunnel', 'nunique'))
tab['예상 입구 사진(×0.41)'] = (tab['입구x날'] * 0.41).round().astype(int)
print('=== 새로 받을 수 있는 장면 (이미 받은 172장의 장면 제외) ==='); print(tab.to_string())
print('\n=== 달별 (새 장면) ==='); print(new.groupby(['도로', 'month']).size().unstack(fill_value=0).to_string())
print('\n=== 연도별 (새 장면) ==='); print(new.groupby(['도로', new.captured_at.dt.year]).size().unstack(fill_value=0).to_string())
print('\n=== 시간대별 (새 장면) ==='); print(new.groupby(['도로', 'hour']).size().unstack(fill_value=0).to_string())


## 4. (확인한 뒤) 사진 받기

- 새 장면마다 입구에서 가장 가까운 1장. 사진 링크는 시간이 지나면 만료돼서, 받기 직전에 사진 번호로 새 링크를 다시 받아요.
- 저장: `data/tunnel_photos_추가/` (사진) + `data/tunnel_photos_추가_meta.csv`
- 받은 뒤 `사진분류_도구.html`로 1~5 분류 → `사진분류_이름.csv`를 Claude에게
- `ROADS`에서 받을 도로를 고르고, `DOWNLOAD = True`로 바꿔 실행


In [ ]:
DOWNLOAD = False
ROADS = ['고속도로']            # 일반 도로터널도 받으려면 ['고속도로', '일반 도로터널']
if DOWNLOAD:
    IMG = OUT / 'tunnel_photos_추가'; IMG.mkdir(exist_ok=True)
    todo = new[new['도로'].isin(ROADS)]
    ok = []
    for n, (_, r) in enumerate(todo.iterrows()):
        fp = IMG / f'{r.image_id}.jpg'
        if fp.exists(): ok.append(r.image_id); continue
        try:
            u = requests.get(f'https://graph.mapillary.com/{r.image_id}',
                             params={'access_token': TOKEN, 'fields': 'thumb_1024_url'}, timeout=30).json().get('thumb_1024_url')
            if u:
                fp.write_bytes(requests.get(u, timeout=30).content); ok.append(r.image_id)
        except Exception as e:
            print(r.image_id, e)
        if n % 50 == 0: print(n, '/', len(todo))
        time.sleep(0.05)
    meta = todo[todo.image_id.isin(ok)].drop(columns=['이미받음', '새 장면'])
    meta.to_csv(OUT / 'tunnel_photos_추가_meta.csv', index=False, encoding='utf-8-sig')
    print('받은 사진', len(ok), '/', len(todo), '→', IMG)
